In [0]:
# COMMAND ----------
%load_ext autoreload
%autoreload 2

import sys
# Wyłączenie generowania starych plików cache .pyc
sys.dont_write_bytecode = True

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC # 🕵️ Advanced Performance Auditor (Serverless Safe)
# MAGIC Notatnik orkiestrujący heurystyczne audyty wydajnościowe i FinOps dla warstwy **Bronze** za pomocą silnika `PerformanceEngine`.

In [0]:
# COMMAND ----------
import sys
import os

# Dynamiczne podpięcie katalogu głównego projektu do sys.path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Importy komponentów silnika APM
from src.auditor.engine import PerformanceEngine
from src.readers.dataframe_reader import DataFrameExplainReader
from src.policies.policy_manager import PolicyManager
from src.context.environment_provider import EnvironmentProvider
from src.suggestions.remediation_engine import RemediationEngine
from src.finops.cost_translator import CostTranslator
from src.reporters.console_reporter import ConsoleReporter

# Importy wszystkich reguł detekcji
from src.rules.physical_rules import SmallFilesRule, MissedBroadcastRule, OverPartitioningRule
from src.rules.query_rules import TypeCastingRule

# 1. Inicjalizacja globalnego stosu technologicznego (Wstrzykiwanie zależności)
policy_mgr = PolicyManager()
env_prov = EnvironmentProvider(spark)
remedy_eng = RemediationEngine()
cost_trans = CostTranslator(policy_mgr.get_policy("finops"))
reporter = ConsoleReporter()

# 2. Kompletowanie pełnego zestawu aktywnych detektywów wydajnościowych
active_rules = [
    SmallFilesRule(),
    MissedBroadcastRule(),
    TypeCastingRule(),
    OverPartitioningRule()
]

print("⚙️ [APM] Framework został zainicjalizowany. Zarejestrowano 4 reguły heurystyczne.")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: Ingest telemetrii BESS (ETL-01)...")

TABLE_BESS = "workspace.bronze.bess_telemetry_small_files"
BESS_SOURCE_PATH = "/Volumes/workspace/default/weather_data/raw_source/bess_telemetry_raw"

# Ładowanie danych in-memory
df_bess = spark.read.table(TABLE_BESS)

# Czytnik dostaje DataFrame i ścieżkę do plików fizycznych
reader_bess = DataFrameExplainReader(spark, df=df_bess, source_volume_path=BESS_SOURCE_PATH)

engine_bess = PerformanceEngine(reader_bess, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter)
engine_bess.run_audit(context_name="ETL-01-BESS-Telemetry")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: Wzbogacanie słownikowe (ETL-03)...")

TABLE_ENRICHED = "workspace.bronze.enriched_telemetry_heavy_shuffle"
df_enriched = spark.read.table(TABLE_ENRICHED)

# Przekazujemy df dla schematu oraz table_name dla bezpośredniego zapytania SQL EXPLAIN pod Serverless
reader_enriched = DataFrameExplainReader(spark, df=df_enriched, table_name=TABLE_ENRICHED)

engine_enriched = PerformanceEngine(reader_enriched, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter)
engine_enriched.run_audit(context_name="ETL-03-Station-Enrichment")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: Ingest odczytów PV (ETL-02)...")

TABLE_PV = "workspace.bronze.pv_metrics_string_nightmare"
df_pv = spark.read.table(TABLE_PV)

# Bezpieczne przekazanie parametrów do strukturalnej analizy schematów
reader_pv = DataFrameExplainReader(spark, df=df_pv, table_name=TABLE_PV)

engine_pv = PerformanceEngine(reader_pv, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter)
engine_pv.run_audit(context_name="ETL-02-PV-Ingest-Casting")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: Logi inwerterów (ETL-04)...")

TABLE_INVERTER = "workspace.bronze.inverter_logs_partition_nightmare"
df_inverter = spark.read.table(TABLE_INVERTER)

reader_inverter = DataFrameExplainReader(spark, df=df_inverter, table_name=TABLE_INVERTER)

engine_inverter = PerformanceEngine(reader_inverter, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter)
engine_inverter.run_audit(context_name="ETL-04-Inverter-OverPartitioning")